In [ ]:
import numpy as np
from sklearn.ensemble import VotingClassifier 
from sklearn.linear_model import LogisticRegression 
from sklearn.svm import SVC 
from sklearn.tree import DecisionTreeClassifier 
from scipy.io import arff
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [ ]:
data, meta = arff.loadarff('dataSet.arff')
df = pd.DataFrame(data)
df = df.astype(int)
X = df.drop('Result', axis=1)
Y = df['Result']

label_encoder = LabelEncoder()
Y_integer = label_encoder.fit_transform(Y)

# Y_binary = (Y + 1) // 2
# print(Y_binary[0:5])
# print(Y.head())


from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y_integer, test_size = 0.2, random_state = 2023)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
lr=LogisticRegression(max_iter=5000, C=0.1)
svm = SVC(kernel="rbf", probability=True)
rf = RandomForestClassifier(max_depth=20, n_estimators=200)
xgb = XGBClassifier(learning_rate=0.2, n_estimators=200)

voting_clf = VotingClassifier(estimators=[('lr', lr), ('svm', svm), ('rf', rf), ('xgb', xgb)], voting="soft", verbose=1)

voting_clf.fit(X_train, Y_train)

In [ ]:
print("Accuracy on test: " + str(voting_clf.score(X_test, Y_test)))
print("Accuracy on train: "+ str(voting_clf.score(X_train, Y_train)))

In [ ]:
param_grid = {
    'lr__C': [0.1, 1, 10],
    'svm__C': [0.1, 1, 10],
    'xgb__max_depth': [5, 10, 15, 20],
    'rf__min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(estimator=voting_clf, param_grid=param_grid, cv=5, scoring='accuracy', verbose = 1, n_jobs=-1)
grid_search.fit(X_train, Y_train)

In [ ]:
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print('Best parameters: ', best_params)
print('Best model', best_model)

In [ ]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(Y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
import joblib
joblib.dump(best_model, 'vclf.pkl')

In [ ]:
import MetricFunctions
MetricFunctions.printAccuracy(best_model, X_train, X_test, Y_train, Y_test)
MetricFunctions.printLoss(best_model, X_train, X_test, Y_train, Y_test)

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)
y_score = y_prob[:, 1]

In [ ]:
MetricFunctions.printAUCandPvalues(Y_test, y_score)

In [ ]:
MetricFunctions.printAUCPRandF1Scores(Y_test, y_pred, y_score)

In [ ]:
MetricFunctions.plotROCCurve(Y_test, y_score)

In [ ]:
MetricFunctions.plotDetCurve(best_model, X_test, Y_test)

In [ ]:
MetricFunctions.printConfMatrix(Y_test, y_pred)
MetricFunctions.plotConfMatrix(Y_test, y_pred)

In [ ]:
import scikitplot as skplt
import matplotlib.pyplot as plt
skplt.metrics.plot_precision_recall(Y_test, y_prob)
plt.show()

In [ ]:
rf_classifier = voting_clf.named_estimators_['rf']
MetricFunctions.plotFeatureImportance(rf_classifier, X)